In [1]:
#undef __noinline__

# GPU Ripple Pattern

This notebook demonstrates GPU parallelism using a 2D ripple pattern rendered on the GPU.

## How It Works

Each pixel is assigned to one GPU thread using a 2D launch grid. The thread computes a
grayscale value based on the pixel's distance from the image centre, producing
concentric rings that fade with distance.

### The Kernel

The kernel maps thread and block indices to a 2D pixel position:
```cpp
int x = threadIdx.x + blockIdx.x * blockDim.x;
int y = threadIdx.y + blockIdx.y * blockDim.y;
```

It then computes the Euclidean distance from the centre and feeds it into a cosine wave:
```
grey = 128 + 127 * cos(d / 10) / (d / 10 + 1)
```

- `d` — distance from the image centre  
- The denominator `(d/10 + 1)` attenuates the wave amplitude with distance, giving a natural fall-off

In [2]:
#define DIM 1024

__global__ void ripple(unsigned char *fb) {
    int x = threadIdx.x + blockIdx.x * blockDim.x;
    int y = threadIdx.y + blockIdx.y * blockDim.y;
    int idx = x + y * blockDim.x * gridDim.x;

    // Vector from pixel to image centre
    float dx = x - DIM / 2.0f;
    float dy = y - DIM / 2.0f;
    float d_sq = dx * dx + dy * dy;

    // Hardware sqrt and cos via PTX intrinsics
    float dist, cos_val;
    asm("sqrt.rn.f32 %0, %1;" : "=f"(dist) : "f"(d_sq));
    float ang = dist / 10.0f;
    asm("cos.approx.f32 %0, %1;" : "=f"(cos_val) : "f"(ang));

    unsigned char grey = (unsigned char)(128.0f + 127.0f * cos_val / (ang + 1.0f));

    fb[idx * 4 + 0] = grey;
    fb[idx * 4 + 1] = grey;
    fb[idx * 4 + 2] = grey;
    fb[idx * 4 + 3] = 255;
}

### Launch Configuration

The image is `1024 × 1024` pixels, divided into `16 × 16` thread blocks.
The kernel launches `64 × 64 = 4096` blocks with `256` threads each — one thread per pixel.

In [3]:
#include "headers/stb_image_write.h"

unsigned char *dev_ptr;
cudaMalloc(&dev_ptr, DIM * DIM * 4);

dim3 blocks(DIM / 16, DIM / 16);
dim3 threads(16, 16);
ripple<<<blocks, threads>>>(dev_ptr);
cudaDeviceSynchronize();

unsigned char host_buf[DIM * DIM * 4];
cudaMemcpy(host_buf, dev_ptr, DIM * DIM * 4, cudaMemcpyDeviceToHost);

stbi_write_png("images/ripple.png", DIM, DIM, 4, host_buf, DIM * 4);
cudaFree(dev_ptr);

JIT session error: Symbols not found: [ stbi_write_png ]
Failed to execute via ::process:Failed to materialize symbols: { (main, { threads, host_buf, blocks, $.incr_module_7.__inits.0, __orc_init_func.incr_module_7, dev_ptr }) }


Error: : Compilation error! JIT session error: Symbols not found: [ stbi_write_png ]
Failed to execute via ::process:Failed to materialize symbols: { (main, { threads, host_buf, blocks, $.incr_module_7.__inits.0, __orc_init_func.incr_module_7, dev_ptr }) }


In [ ]:
#include "headers/display.hpp"

im::image ripple_image("images/ripple.png");
xcpp::display(ripple_image);